# EAPS 507 — Applying Functions to Xarray Data
## Vectorization, broadcasting, and `xr.apply_ufunc()`

**Estimated time:** 90 minutes  
**Context:** atmospheric temperature on pressure levels

### Learning objectives

By the end of this tutorial, you should be able to:

1. explain vectorization and broadcasting in plain language;
2. distinguish elementwise operations from operations that use a core dimension;
3. write a numerical function that works on array-like inputs;
4. use `xr.apply_ufunc()` while preserving Xarray coordinates and metadata;
5. use `input_core_dims` and `output_core_dims` for profile-based calculations;
6. decide when direct Xarray arithmetic, `apply_ufunc()`, or `map_blocks()` is appropriate.

### Class rhythm

Each section follows **explain → predict → run → interpret → your turn**. Work with a partner for the prediction and “your turn” prompts, but write your own explanation.

| Section | Topic | Time |
|---|---|---:|
| 1 | Setup and labeled data | 10 min |
| 2 | Elementwise functions | 15 min |
| 3 | Broadcasting and vectorization | 15 min |
| 4 | `apply_ufunc()` | 20 min |
| 5 | Core dimensions | 20 min |
| 6 | 2-minute example + wrap-up | 10 min |


## 1. Setup and labeled data — 10 minutes

We will first try to open the ERA5 file used in class. If it is unavailable, the next cell constructs a small dataset with the same essential dimensions. This makes the tutorial portable while keeping the atmospheric-science context.


In [1]:
from pathlib import Path

import numpy as np
import xarray as xr

xr.set_options(keep_attrs=True, display_expand_data=False)

DATA_PATH = Path("ERA5_T_Rh_202504.nc")


In [ ]:
temp_rh_main.chun

In [5]:
if DATA_PATH.exists():
    temp_rh_main = (
        xr.open_mfdataset(DATA_PATH)
        .rename(
            {
                "valid_time": "time",
                "pressure_level": "level",
                "latitude": "lat",
                "longitude": "lon",
            }
        )
        .isel(time=slice(0, 5))#.chunks({'time':1})
        .load()
    )
    print("Using the ERA5 class file.")
else:
    rng = np.random.default_rng(507)
    time = np.arange("2025-04-01T00", "2025-04-01T12", dtype="datetime64[3h]")
    level = np.array([1000, 925, 850, 700, 500, 300])
    lat = np.array([39.5, 40.0, 40.5])
    lon = np.array([-87.5, -87.0, -86.5, -86.0])

    base_profile = 292 - 0.065 * (1000 - level)
    temperature = (
        base_profile[None, :, None, None]
        + rng.normal(0, 1.2, size=(time.size, level.size, lat.size, lon.size))
    )
    relative_humidity = rng.uniform(
        35, 95, size=(time.size, level.size, lat.size, lon.size)
    )

    temp_rh_main = xr.Dataset(
        data_vars={
            "t": (
                ("time", "level", "lat", "lon"),
                temperature,
                {"long_name": "air temperature", "units": "K"},
            ),
            "r": (
                ("time", "level", "lat", "lon"),
                relative_humidity,
                {"long_name": "relative humidity", "units": "%"},
            ),
        },
        coords={"time": time, "level": level, "lat": lat, "lon": lon},
        attrs={"description": "Synthetic EAPS 507 atmospheric dataset"},
    )
    print("Class file not found; using the portable sample dataset.")

temp_rh_main


getfattr: /home/ktquagra/EAPS_507_F2026/Students/Week: No such file or directory
getfattr: 6/ERA5_T_Rh_202504.nc: No such file or directory


Using the ERA5 class file.


<xarray.Dataset> Size: 78MB
Dimensions:  (time: 5, level: 27, lat: 201, lon: 361)
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
Data variables:
    r        (time, level, lat, lon) float32 39MB 34.35 31.89 ... 52.28 52.84
    t        (time, level, lat, lon) float32 39MB 277.1 277.5 ... 195.0 194.9
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-29T00:00 GRIB to CDM+CF via cfgrib-0.9.1...

### Read the data model

An Xarray `DataArray` combines values with:

- **dimensions**: axis names such as `time`, `level`, `lat`, and `lon`;
- **coordinates**: labels along those dimensions;
- **attributes**: descriptive metadata such as units.

The labels are central to how Xarray aligns and broadcasts calculations.


In [6]:
list(temp_rh_main.variables)

['r', 't', 'number', 'time', 'level', 'lat', 'lon', 'expver']

In [10]:
t = temp_rh_main["t"]

print("Dimensions:", t.dims)
print("Shape:", t.shape)
print("Coordinates:", list(t.coords))
print("Units:", t.attrs.get("units"))


Dimensions: ('time', 'level', 'lat', 'lon')
Shape: (4, 6, 3, 4)
Coordinates: ['time', 'level', 'lat', 'lon']
Units: K


### Predict, then test

Before running the next cell, predict the result of selecting one time and one location.

1. Which dimension will remain?

    Time
2. What will the shape be?

    1
3. Will the pressure-level labels be preserved?

    Yes because we didn't pick out what pressure level we wanted.

In [5]:
profile = t.isel(time=0, lat=1, lon=1)
profile


<xarray.DataArray 't' (level: 6)> Size: 48B
294.2 286.3 283.4 272.9 258.2 247.4
Coordinates:
  * level    (level) int64 48B 1000 925 850 700 500 300
    time     datetime64[s] 8B 1988-06-01T08:00:00
    lat      float64 8B 40.0
    lon      float64 8B -87.0
Attributes:
    long_name:  air temperature
    units:      K

### Your turn 1

Select the temperature profile at the **last time**, nearest to **40°N, 87°W**. Use `.sel(..., method="nearest")` for latitude and longitude.


In [12]:
# YOUR TURN 1

#Selecting the temperature profile at the last time near 40,-87
selected_profile = t.isel(time=-1).sel(lat=40.0, lon=-87.0, method="nearest")
selected_profile


<xarray.DataArray 't' (level: 6)> Size: 48B
293.0 285.1 281.7 274.1 256.6 247.7
Coordinates:
  * level    (level) int64 48B 1000 925 850 700 500 300
    time     datetime64[s] 8B 1988-06-01T11:00:00
    lat      float64 8B 40.0
    lon      float64 8B -87.0
Attributes:
    long_name:  air temperature
    units:      K

---
## 2. Elementwise functions — 15 minutes

An **elementwise function** treats every value independently. Converting Kelvin to Celsius is elementwise: each output value depends on one input value at the same position.

NumPy-style arithmetic is already vectorized. We describe the calculation once, and the array library applies it across all values without an explicit Python loop.


In [14]:
def kelvin_to_celsius(temperature_k):
    'Convert an array-like temperature from kelvin to degrees Celsius.'
    return temperature_k - 273.15


t_c = kelvin_to_celsius(t)
t_c.attrs = {"long_name": "air temperature", "units": "degC"}
t_c


<xarray.DataArray 't' (time: 4, level: 6, lat: 3, lon: 4)> Size: 2kB
18.42 20.75 20.57 21.71 18.61 21.07 ... -26.28 -25.34 -24.56 -25.86 -26.7 -27.14
Coordinates:
  * time     (time) datetime64[s] 32B 1988-06-01T08:00:00 ... 1988-06-01T11:0...
  * level    (level) int64 48B 1000 925 850 700 500 300
  * lat      (lat) float64 24B 39.5 40.0 40.5
  * lon      (lon) float64 32B -87.5 -87.0 -86.5 -86.0
Attributes:
    long_name:  air temperature
    units:      degC

### Why did that work without `apply_ufunc()`?

The function uses subtraction, which both NumPy and Xarray define for arrays. When a function is composed entirely of operations Xarray understands, calling it directly is usually the clearest choice.

> **Rule of thumb:** try direct Xarray arithmetic first. Reach for `apply_ufunc()` when you need to wrap a function that does not already understand labeled Xarray objects, or when you must explicitly describe core dimensions.


### Loop versus vectorized thinking

The two calculations below are conceptually equivalent, but the vectorized form is shorter, clearer, and generally faster.


In [7]:
small_values = np.array([280.0, 285.0, 290.0])

# Explicit Python loop
loop_result = []
for value in small_values:
    loop_result.append(value - 273.15)
loop_result = np.array(loop_result)

# Vectorized array expression
vectorized_result = small_values - 273.15

print("Loop result:      ", loop_result)
print("Vectorized result:", vectorized_result)
print("Same result?", np.allclose(loop_result, vectorized_result))


Loop result:       [ 6.85 11.85 16.85]
Vectorized result: [ 6.85 11.85 16.85]
Same result? True


### Student-led test 2

Write a function named `celsius_to_fahrenheit()` using:


$T_F = \frac{9}{5}T_C + 32.$

Apply it directly to `t_c`. Confirm that the output retains all four dimensions and the original coordinates.


In [22]:
# YOUR TURN 2
#Create a function to change the units from degrees celsius to fahrenheit
def celsius_to_fahrenheit(temperature_c):
    return ((9/5)*temperature_c) + 32


t_f = celsius_to_fahrenheit(t_c)
t_f.attrs = {"long_name": "air temperature", "units": "degF"}
print(t_f.dims)
t_f


('time', 'level', 'lat', 'lon')


<xarray.DataArray 't' (time: 4, level: 6, lat: 3, lon: 4)> Size: 2kB
65.15 69.36 69.03 71.08 65.5 69.92 ... -15.3 -13.62 -12.21 -14.55 -16.07 -16.85
Coordinates:
  * time     (time) datetime64[s] 32B 1988-06-01T08:00:00 ... 1988-06-01T11:0...
  * level    (level) int64 48B 1000 925 850 700 500 300
  * lat      (lat) float64 24B 39.5 40.0 40.5
  * lon      (lon) float64 32B -87.5 -87.0 -86.5 -86.0
Attributes:
    long_name:  air temperature
    units:      degF

---
## 3. Broadcasting and vectorization — 15 minutes

**Broadcasting** lets arrays with different dimensions participate in one calculation when their dimensions are compatible.

Potential temperature is


$\theta = T\left(\frac{p_0}{p}\right)^{R_d/c_p}$,


where (T) varies over `time`, `level`, `lat`, and `lon`, while pressure (p) varies only over `level`.


In [9]:
P0_HPA = 1000.0
RD_OVER_CP = 0.286


def potential_temperature(temperature_k, pressure_hpa):
    'Potential temperature in K from temperature and pressure in hPa.'
    return temperature_k * (P0_HPA / pressure_hpa) ** RD_OVER_CP


pressure = temp_rh_main["level"]

print("Temperature dimensions:", t.dims)
print("Pressure dimensions:   ", pressure.dims)


Temperature dimensions: ('time', 'level', 'lat', 'lon')
Pressure dimensions:    ('level',)


### Predict first

What dimensions should the result have when we calculate `potential_temperature(t, pressure)`?

4

Xarray matches `pressure.level` to `t.level`, then broadcasts pressure across `time`, `lat`, and `lon`. It does not rely only on axis position.


In [10]:
theta_direct = potential_temperature(t, pressure)
theta_direct.name = "theta"
theta_direct.attrs = {"long_name": "potential temperature", "units": "K"}

print("Result dimensions:", theta_direct.dims)
print("Result shape:", theta_direct.shape)
theta_direct.isel(time=0, lat=1, lon=1)


Result dimensions: ('time', 'level', 'lat', 'lon')
Result shape: (4, 6, 3, 4)


<xarray.DataArray 'theta' (level: 6)> Size: 48B
294.2 292.7 296.9 302.2 314.8 349.1
Coordinates:
  * level    (level) int64 48B 1000 925 850 700 500 300
    time     datetime64[s] 8B 1988-06-01T08:00:00
    lat      float64 8B 40.0
    lon      float64 8B -87.0
Attributes:
    long_name:  potential temperature
    units:      K

### Coordinate alignment test

Xarray aligns inputs by coordinate labels. Reversing the order of the pressure coordinate should not change the physical answer, because the labels still identify each level.


In [11]:
pressure_reversed = pressure.isel(level=slice(None, None, -1))
theta_reversed_input = potential_temperature(t, pressure_reversed)

xr.testing.assert_allclose(theta_direct, theta_reversed_input)
print("Test passed: Xarray aligned pressure by level labels.")


Test passed: Xarray aligned pressure by level labels.


---
## 6. Two-minute example — 10 minutes

### Independent task

Create a function called `wind_speed(u, v)` that calculates

$V = \sqrt{u^2 + v^2}.$

The cell below supplies small `u` and `v` wind-component arrays. Apply your function with `xr.apply_ufunc()` and answer the interpretation questions.

**This is the part to submit:** export only Section 6 as a PDF, including your completed code, output, and written responses.


In [30]:
u = xr.DataArray(
    [[3.0, 0.0, -5.0], [6.0, 8.0, 0.0]],
    dims=("station", "time"),
    coords={"station": ["A", "B"], "time": ["00Z", "06Z", "12Z"]},
    attrs={"units": "m s-1", "long_name": "eastward wind"},
)

v = xr.DataArray(
    [[4.0, 5.0, 12.0], [8.0, 6.0, -7.0]],
    dims=("station", "time"),
    coords={"station": ["A", "B"], "time": ["00Z", "06Z", "12Z"]},
    attrs={"units": "m s-1", "long_name": "northward wind"},
)

display(u)
display(v)


<xarray.DataArray (station: 2, time: 3)> Size: 48B
3.0 0.0 -5.0 6.0 8.0 0.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
  * time     (time) <U3 36B '00Z' '06Z' '12Z'
Attributes:
    units:      m s-1
    long_name:  eastward wind

<xarray.DataArray (station: 2, time: 3)> Size: 48B
4.0 5.0 12.0 8.0 6.0 -7.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
  * time     (time) <U3 36B '00Z' '06Z' '12Z'
Attributes:
    units:      m s-1
    long_name:  northward wind

In [34]:
# SECTION 6 — COMPLETE AND SUBMIT THIS CELL

# 1. Define the function.
def wind_speed(u_component, v_component):
    return np.sqrt(u_component**2 + v_component**2)


# 2. Apply the function with xr.apply_ufunc().
speed = xr.apply_ufunc(
    wind_speed,
    u,
    v,
    keep_attrs=True,
)

# 3. Add a name and scientifically appropriate metadata.
speed.name = "wind_speed"
speed.attrs = {
    "long_name": "wind speed",
    "units": "m s-1",
}

# 4. Display the result.
speed
print(speed.sel(time="00Z"))
print(speed.sel(time="06Z"))
print(speed.sel(time="12Z"))

<xarray.DataArray 'wind_speed' (station: 2)> Size: 16B
5.0 10.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
    time     <U3 12B '00Z'
Attributes:
    long_name:  wind speed
    units:      m s-1
<xarray.DataArray 'wind_speed' (station: 2)> Size: 16B
5.0 10.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
    time     <U3 12B '06Z'
Attributes:
    long_name:  wind speed
    units:      m s-1
<xarray.DataArray 'wind_speed' (station: 2)> Size: 16B
13.0 7.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
    time     <U3 12B '12Z'
Attributes:
    long_name:  wind speed
    units:      m s-1


### Section 6 interpretation — include in your submission

Answer in complete sentences.

1. What are the dimensions and coordinates of `speed`?

    The dimenstions of speed are 'station' and 'time.' The coordinates are station('A','B') at times 0Z, 6Z, and 12Z.

2. At which station and time is wind speed greatest? Give its value and units.

    The station with the greatest wind speed is Station A with 13 m/s.
3. Why does this elementwise function not require `input_core_dims`?
        
        Since we are only using u and v, and no other elements, we don't need to reduce any dimensions to do this calculation.
        
4. Would `vectorize=True` be necessary for the function you wrote? Explain.

    No, with Xarray, we are able to just use regular operations because they are already vectorized due to NumPy.

## Wrap-up

- **Vectorization** expresses a calculation for arrays instead of writing a Python loop over individual values.
- **Broadcasting** expands compatible arrays across dimensions and, in Xarray, uses dimension names.
- **Direct Xarray arithmetic** is best when the function already supports labeled arrays.
- **`xr.apply_ufunc()`** adapts unlabeled array functions while restoring Xarray dimensions and coordinates.
- **Core dimensions** are the axes a function consumes as a unit, such as an entire vertical profile.

### Exit ticket

In one sentence each:

1. What is the difference between a broadcast dimension and a core dimension?
    
    It's preserved across calculations and makes the labels match. A core dimension is consumed.
2. When would you use `vectorize=True`?

    When you are using a function that only accepts scalar inputs.
    
3. Why should metadata be checked after applying a custom function?

    You can accidentally change attributes in a function, so making sure your attributes are okay after your function keeps everything correct.
    